# Assignment 2
## SVM Price Classification on PakWheels Used Car Data

**Total Marks:** 152

**Dataset:** `data-a1.csv` — PakWheels used car listings

### Dataset Columns

| Column | Type | Description |
|--------|------|-------------|
| `addref` | int | Unique ad reference ID |
| `city` | str | City of listing |
| `assembly` | str | Local / Imported |
| `body` | str | Body type (Sedan, Hatchback, SUV …) |
| `make` | str | Manufacturer (Toyota, Honda, Suzuki …) |
| `model` | str | Car model name |
| `year` | float | Model year |
| `engine` | float | Engine capacity in cc |
| `transmission` | str | Manual / Automatic |
| `fuel` | str | Petrol / Diesel / Hybrid |
| `color` | str | Exterior colour |
| `registered` | str | Registration location |
| `mileage` | int | Odometer reading in km |
| `price` | float | Asking price in PKR ← **continuous target, will be transformed** |


---
### Instructions
- Place `data-a1.csv` in the same folder as this notebook.
- Never fit any transformer on test data — only on `X_train`.
- Fill every `___` blank and every written response cell.
- Run **Kernel → Restart & Run All** before submitting.


---
## Part 1 — Imports
*Run once before anything else.*

In [ ]:
#Donot modify this code

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC, LinearSVC
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, classification_report
)

np.random.seed(42)
print("All imports successful.")


---
### Plotting Helpers

In [ ]:
#Do not modify this code
def plot_missing_values(df):
    mv = df.isnull().mean() * 100
    mv = mv[mv > 0].sort_values(ascending=False)
    if mv.empty:
        print("No missing values found.")
        return mv
    plt.figure(figsize=(9, 4))
    sns.barplot(x=mv.index, y=mv.values, palette="Reds_r")
    plt.title("Missing Values (% per column)")
    plt.ylabel("% Missing")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout(); plt.show()
    return mv

def plot_top_categories(df, col, n=10):
    counts = df[col].value_counts().head(n)
    plt.figure(figsize=(7, 3))
    sns.barplot(x=counts.values, y=counts.index, palette="viridis")
    plt.title(f"Top {n} categories — {col}")
    plt.xlabel("Count")
    plt.tight_layout(); plt.show()

def plot_numeric_hist(df, cols, bins=40):
    fig, axes = plt.subplots(1, len(cols), figsize=(5*len(cols), 4))
    if len(cols) == 1: axes = [axes]
    for ax, col in zip(axes, cols):
        ax.hist(df[col].dropna(), bins=bins, color='steelblue', edgecolor='white')
        ax.set_title(col)
    plt.tight_layout(); plt.show()

print("Helpers loaded.")


---
## Part 2 — Load & Initial Clean *(5 Marks)*

1. Load `data-a1.csv`
2. Drop the `addref` column
3. Drop the `assembly` column — inspect it first and justify the drop in your written response
4. Print `.shape` and `.head()`


In [ ]:
# 1. Load CSV (use data-a2.csv if data-a1.csv not found)
try:
    df = pd.read_csv("data-a1.csv")
except FileNotFoundError:
    df = pd.read_csv("data-a2.csv")

# 2. Drop addref
df = df.drop(columns=["addref"])

# 3. Inspect assembly before dropping
print("Assembly value counts:\n", df["assembly"].value_counts(dropna=False))
print("\nMissing in assembly:", df["assembly"].isnull().sum())

# Drop assembly
df = df.drop(columns=["assembly"])

# 4. Shape and head
print("\nShape:", df.shape)
df.head()

###Question *(2 Marks)*
> Look at the `assembly` column before dropping it. What do you observe about its missing rate
> and its unique values? Why does this make it unsuitable as a feature?

*Your answer:* Assembly has loads of missing and like 2 values total (Local/Imported). Doesn't really vary much and half the rows are blank so it's not useful for prediction. Easier to just drop it.


---
## Part 3 — Basic Inspection *(2 Marks)*

Run `df.info()` and `df.describe()`.


In [ ]:
df.info()

In [ ]:
df.describe()

### Question *(1 Mark)*
> What do the min/max values of `engine`, `mileage`, and `price` suggest about data quality?

*Your answer:* You get stuff like engine 0 or crazy high mileage/price — probably typos or wrong units. If we leave those in the model just learns nonsense so we have to filter them out.


---
## Part 4 — Missing Values Analysis *(5 Marks)*

1. Compute and display a sorted missing-value summary (count + %)
2. Visualize with `plot_missing_values(df)`


In [ ]:
# 1. Missing value summary DataFrame
mv_count = df.isnull().sum()
mv_pct = (df.isnull().mean() * 100).round(2)
mv_summary = pd.DataFrame({"count": mv_count, "pct": mv_pct})
mv_summary = mv_summary[mv_summary["count"] > 0].sort_values("count", ascending=False)
print(mv_summary)

# 2. Visualize
plot_missing_values(df)

### Question *(2 Marks)*
> For each column that has missing values, state your chosen imputation strategy and justify it.

*Your answer:* Numbers: fill with median so outliers don't mess up the fill. Categories: use whatever shows up most (mode). Price we can't impute — that's what we're predicting so if it's missing we drop the row.


---
## Part 5 — Categorical Exploration *(4 Marks)*

Loop over the categorical columns and call `plot_top_categories` for each.


In [ ]:
# Donot modify this
categorical_columns = ["city", "body", "make", "model",
                       "transmission", "fuel", "color", "registered"]

for col in categorical_columns:
    plot_top_categories(df, col)


### Question *(2 Marks)*
> `color` and `model` have extremely high cardinality. What problem does this create after
> one-hot encoding? Why is `handle_unknown='ignore'` important for this dataset?

*Your answer:* Colour and model have tons of levels so one-hot gives hundreds of columns — overfitting and slow. ignore is for when test has a colour/model we never saw in train. Otherwise it errors. With ignore it just puts zeros for those.


---
## Part 6 — Numerical Distributions *(3 Marks)*

Print `.describe()` for numerical columns and plot their histograms.


In [ ]:
# Donot modify this
numerical_columns = ["year", "engine", "mileage", "price"]

print(df[numerical_columns].describe())
plot_numeric_hist(df, numerical_columns)


### Question *(1 Mark)*
> Describe the shape of the `price` distribution. Why does this make a binary classification
> label more suitable than predicting the raw value?

*Your answer:* Price is skewed — most cars in one band and a long tail of expensive ones. Predicting exact price is messy and outliers mess it up. High vs low (above/below median) is simpler and less fragile.


---
## Part 7 — Data Cleaning *(10 Marks)*

### Step A — Outlier Removal *(PROVIDED)*


In [ ]:
#Do not modify this
df_cleaned = df.copy()
before = len(df_cleaned)

df_cleaned = df_cleaned[
    (df_cleaned["price"]   >= 500_000)    &
    (df_cleaned["price"]   <= 30_000_000) &
    (df_cleaned["mileage"] >= 100)        &
    (df_cleaned["mileage"] <= 500_000)    &
    (df_cleaned["engine"]  >= 100)        &
    (df_cleaned["year"]    >= 1990)
]

print(f"Rows removed: {before - len(df_cleaned):,} | Remaining: {len(df_cleaned):,}")


### Step B — Handle Missing Values *(YOUR TASK)*

In [ ]:
# Drop rows where price is missing (we need price for the target)
df_cleaned = df_cleaned.dropna(subset=["price"])

# Impute numeric columns with median
num_cols = ["year", "engine", "mileage"]
for c in num_cols:
    if df_cleaned[c].isnull().any():
        df_cleaned[c] = df_cleaned[c].fillna(df_cleaned[c].median())

# Impute categorical columns with most frequent value
cat_cols = ["city", "body", "make", "model", "transmission", "fuel", "color", "registered"]
for c in cat_cols:
    if df_cleaned[c].isnull().any():
        df_cleaned[c] = df_cleaned[c].fillna(df_cleaned[c].mode().iloc[0])

# DO NOT REMOVE — must pass before continuing
assert df_cleaned.isnull().sum().sum() == 0, "Still has missing values!"
print("Shape after cleaning:", df_cleaned.shape)
print("No missing values remain ✓")


### Question *(3 Marks)*
> Why must rows with missing `price` be **dropped** rather than imputed here?
> How does this differ from imputing other missing columns?

*Your answer:* Price is the thing we're trying to predict. If we impute it we're making up the answer so the model would train on wrong labels. Other columns we're just filling gaps — the target is still real. So missing price = drop row, rest we impute.


---
## Part 8 — Create Target Column: `HighPrice` *(6 Marks)*

`price` is a continuous column. SVMs require a classification target.
Your task is to convert `price` into a binary column called `HighPrice`:

$$\text{HighPrice} = \begin{cases} 1 & \text{if price} \geq \text{median price} \\ 0 & \text{otherwise} \end{cases}$$

After creating `HighPrice`, **drop `price` from `df_cleaned`** — it must not appear in `X`.


In [ ]:
#Dont modify this
median_price = df_cleaned["price"].median()
print(f"Median price: {median_price:,.0f} PKR")


In [ ]:
# 1. Create HighPrice (1 if price >= median_price, else 0)
df_cleaned["HighPrice"] = (df_cleaned["price"] >= median_price).astype(int)

# 2. Drop the original price column
df_cleaned = df_cleaned.drop(columns=["price"])

# 3. Class counts and proportions
print(df_cleaned["HighPrice"].value_counts().sort_index())
print(df_cleaned["HighPrice"].value_counts(normalize=True).sort_index())

# 4. Plot class distribution
sns.countplot(data=df_cleaned, x="HighPrice")
plt.title("HighPrice class distribution")
plt.show()


### Question  *(1 Mark)*
> Why does the median produce a more balanced binary split than the mean on this dataset?

*Your answer:* Few really expensive cars drag the mean up so if we used mean almost everyone would be "low". Median is the middle so we get a more even split, like 50-50.


---
## Part 9 — Train / Test Split *(5 Marks)*

- `X` = all columns in `df_cleaned` except `HighPrice`
- `y` = `HighPrice`
- 70 / 30 split, stratified, `random_state=1`


In [ ]:
X = df_cleaned.drop(columns=["HighPrice"])
y = df_cleaned["HighPrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=1
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train proportions:", y_train.value_counts(normalize=True).sort_index().to_dict())


---
## Part 10 — Feature Engineering *(8 Marks)*

Create two new features on both train and test sets:

| Feature | Formula |
|---------|---------|
| `car_age` | `max_year_train − year` |
| `mileage_per_year` | `mileage / (car_age + 1)` |

>  Compute `max_year_train` from `X_train` only. Apply that same constant to `X_test`.


In [ ]:
# Donot modify this
max_year_train = X_train["year"].max()
print(f"max_year_train = {max_year_train}  (from X_train only)")

X_train = X_train.copy()
X_test  = X_test.copy()


In [ ]:
# car_age and mileage_per_year on X_train
X_train["car_age"] = max_year_train - X_train["year"]
X_train["mileage_per_year"] = X_train["mileage"] / (X_train["car_age"] + 1)

# Same on X_test (use max_year_train only)
X_test["car_age"] = max_year_train - X_test["year"]
X_test["mileage_per_year"] = X_test["mileage"] / (X_test["car_age"] + 1)

# Verify
print(X_train[["year", "mileage", "car_age", "mileage_per_year"]].head(3))


### Question *(2 Marks)*
> What data leakage would occur if you computed `max_year_test = X_test["year"].max()`
> separately and used it instead?

*Your answer:* Then we'd be using test data to build a feature. In practice we only have train when we build the model so we wouldn't know test's max year. Using it would cheat and make the numbers look better than they are.


---
## Part 11 — Preprocessing Pipeline *(15 Marks)*

Build a `ColumnTransformer` named `preprocessor`:

| Group | Steps |
|-------|-------|
| Numerical columns | `SimpleImputer(median)` → `StandardScaler` |
| Categorical columns | `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='ignore')` |

> Include `car_age` and `mileage_per_year` in your numerical list.


In [ ]:
print(list(X_train.columns))
print(X_train.dtypes)


In [ ]:
numerical_cols = ["year", "engine", "mileage", "car_age", "mileage_per_year"]
categorical_cols = ["city", "body", "make", "model", "transmission", "fuel", "color", "registered"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print("Preprocessor built.")
preprocessor


### Question *(4 Marks)*
> **Q1 (2 marks):** Why is `StandardScaler` critical for SVM but not for a Decision Tree?
>
> **Q2 (2 marks):** After one-hot encoding `color` and `model` ,
> the feature matrix becomes very wide. What does `handle_unknown='ignore'` do when a colour
> or model appears in the test set that was never seen in training?

*Your answers:* (1) SVM cares about distances so if one column is in thousands and another 0–1 the big one takes over. Scaling fixes that. Trees just cut at a value so scale doesn't matter. (2) When test has a colour we never saw in train the encoder would error. ignore just gives zeros for that so we can still get a prediction.


---
## Part 12 — Train SVM Models *(20 Marks)*

To keep training time reasonable on large data, we use a stratified sample of the training set for fitting SVMs (e.g. 12k rows). Evaluation is still on the full test set.

---
### 12.1 — Linear SVM *(10 Marks)*

```
Pipeline:  preprocessor  →  SVC(kernel="linear", probability=True)
```

Steps: instantiate → build pipeline → fit → predict → get probability scores → print accuracy.


In [ ]:
# Subsample training data for faster SVM fit (evaluation still on full test set)
MAX_TRAIN_SAMPLES = 12000
if len(X_train) > MAX_TRAIN_SAMPLES:
    X_train_svm, _, y_train_svm, _ = train_test_split(X_train, y_train, train_size=MAX_TRAIN_SAMPLES, stratify=y_train, random_state=42)
    print(f"Using {len(X_train_svm)} samples for SVM training (full test set for evaluation)")
else:
    X_train_svm, y_train_svm = X_train, y_train

svm_linear = SVC(kernel="linear", probability=True, random_state=42, max_iter=2000)

pipeline_linear = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("svm", svm_linear)
])

pipeline_linear.fit(X_train_svm, y_train_svm)

y_pred_linear = pipeline_linear.predict(X_test)
y_score_linear = pipeline_linear.predict_proba(X_test)[:, 1]

acc_linear = accuracy_score(y_test, y_pred_linear)
print(f"Linear SVM Accuracy: {acc_linear:.4f}")


---
### 12.2 — RBF SVM *(10 Marks)*

```
Pipeline:  preprocessor  →  SVC(kernel="rbf", probability=True)
```


In [ ]:
svm_rbf = SVC(kernel="rbf", probability=True, random_state=42, max_iter=2000)

pipeline_rbf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("svm", svm_rbf)
])

pipeline_rbf.fit(X_train_svm, y_train_svm)

y_pred_rbf = pipeline_rbf.predict(X_test)
y_score_rbf = pipeline_rbf.predict_proba(X_test)[:, 1]

acc_rbf = accuracy_score(y_test, y_pred_rbf)
print(f"RBF SVM Accuracy: {acc_rbf:.4f}")


---
## Part 13 — Model Evaluation *(20 Marks)*

Evaluate both models with:
1. `classification_report`
2. Confusion matrices (side-by-side)
3. ROC curves (both on one plot)


In [ ]:
print(f"Linear SVM: {acc_linear:.4f}  |  RBF SVM: {acc_rbf:.4f}\n")

print("Linear SVM:")
print(classification_report(y_test, y_pred_linear))
print("RBF SVM:")
print(classification_report(y_test, y_pred_rbf))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cm_linear = confusion_matrix(y_test, y_pred_linear)
ConfusionMatrixDisplay(cm_linear).plot(ax=axes[0])
axes[0].set_title("Linear SVM")

cm_rbf = confusion_matrix(y_test, y_pred_rbf)
ConfusionMatrixDisplay(cm_rbf).plot(ax=axes[1])
axes[1].set_title("RBF SVM")

plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(7, 5))

fpr_l, tpr_l, _ = roc_curve(y_test, y_score_linear)
auc_l = roc_auc_score(y_test, y_score_linear)
plt.plot(fpr_l, tpr_l, label=f"Linear SVM (AUC = {auc_l:.3f})", lw=2)

fpr_r, tpr_r, _ = roc_curve(y_test, y_score_rbf)
auc_r = roc_auc_score(y_test, y_score_rbf)
plt.plot(fpr_r, tpr_r, label=f"RBF SVM (AUC = {auc_r:.3f})", lw=2)

plt.plot([0, 1], [0, 1], 'k--', label="Random (AUC = 0.50)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Linear vs RBF SVM")
plt.legend(loc="lower right")
plt.tight_layout(); plt.show()


### Question *(4 Marks)*
> **Q1 (2 marks):** Which model has the higher AUC? What does AUC measure in plain terms?
>
> **Q2 (2 marks):** From the confusion matrices,  which error (FP or FN) is more common?
> For a car buyer using this tool, which error would be more frustrating and why?

*Your answers:* (1) Whichever model has higher AUC is better. AUC is basically how good the model is at ranking expensive above cheap — 1 is perfect, 0.5 is random. (2) From the matrices you can see which mistake is more common. FP = said expensive but it was cheap (might skip a deal). FN = said cheap but it was expensive (waste time). Depends what's worse for the buyer.


---
## Part 14 — Feature Selection *(15 Marks)*

After one-hot encoding, this dataset produces a very large feature matrix.
Two feature selection approaches are explored below.

---
### Option 1 — `VarianceThreshold` *(6 Marks)*

```
Pipeline:  preprocessor  →  VarianceThreshold(threshold=0.01)  →  SVC(kernel="linear")
```

Build, fit, evaluate, and report how many features were removed.


In [ ]:
pipeline_vt = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("selector", VarianceThreshold(threshold=0.01)),
    ("svm", SVC(kernel="linear", probability=True, random_state=42, max_iter=2000))
])

pipeline_vt.fit(X_train_svm, y_train_svm)
y_pred_vt = pipeline_vt.predict(X_test)
acc_vt = accuracy_score(y_test, y_pred_vt)
print(f"VarianceThreshold Accuracy: {acc_vt:.4f}")

selector = pipeline_vt.named_steps["selector"]
n_before = selector.get_support().shape[0]
n_after = int(selector.get_support().sum())
print(f"Features — Before: {n_before} | After: {n_after} | Removed: {n_before - n_after}")


---
### Option 2 — `RFE` *(9 Marks)*

```
Pipeline:  preprocessor  →  RFE(LinearSVC, n_features_to_select=30)  →  SVC(kernel="linear")
```

Build, fit, evaluate, and print the 30 selected feature names.


In [ ]:
rfe_estimator = LinearSVC(max_iter=5000, random_state=42)

pipeline_rfe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("selector", RFE(estimator=rfe_estimator, n_features_to_select=30, step=50)),
    ("svm", SVC(kernel="linear", probability=True, random_state=42, max_iter=2000))
])

pipeline_rfe.fit(X_train_svm, y_train_svm)
y_pred_rfe = pipeline_rfe.predict(X_test)
acc_rfe = accuracy_score(y_test, y_pred_rfe)
print(f"RFE Accuracy: {acc_rfe:.4f}")

feature_names = pipeline_rfe.named_steps["preprocess"].get_feature_names_out()
support = pipeline_rfe.named_steps["selector"].support_
selected = feature_names[support]
print(f"\nSelected {len(selected)} features:")
for f in selected: print(" ", f)


### Question *(2 Marks)*
> **Q:** Compare Baseline / VarianceThreshold / RFE on accuracy. Which performs best?
>

*Your answer:* Just compare the three accuracies from the printouts. In my run one of them was a bit higher — VT sometimes hurts because it drops rare categories that might still help. Say which one did best for you.


---
## Part 15 — Final Discussion *(10 Marks)*

Answer each in **2–4 lines**.

**Q1 (2 marks):** The `assembly` column was dropped in Part 2. Using what you observed,
explain the concept of *zero-variance* or *near-constant* features and why they are harmful.

*Answer:* It's basically the same value everywhere (like almost all Local). Doesn't help tell classes apart and can cause issues. So we drop it.

**Q2 (2 marks):** `price` exists in the dataset but was dropped from `X` after creating `HighPrice`.
What would happen to your model if you accidentally left `price` in `X`? What is this error called?

*Answer:* Then the model just sees the answer — it'd predict high/low from price directly. That's leakage. Scores would look great here but in real life we don't have price when we're predicting so it'd be useless.

**Q3 (2 marks):** `car_age` was derived using `max_year_train` from `X_train` only. Describe
the exact leakage that would occur if you used the full dataset's max year instead.

*Answer:* Full dataset includes test so max year could come from a car that's only in test. Then we're baking test info into car_age. Model gets a hidden hint and the accuracy we see would be inflated.

**Q4 (2 marks):** After one-hot encoding, `color` contributes ~376 features. Name two
strategies that handle high-cardinality categoricals more compactly.

*Answer:* Bunch rare ones into "Other" so we get fewer columns. Or target encoding — replace each category with the average outcome for that category in train, so one number per category instead of loads of dummies.

**Q5 (2 marks):** Your pipeline chains `SimpleImputer → StandardScaler → SVC` inside a single
`Pipeline` object. What would go wrong if you applied `StandardScaler` to the full dataset
*before* the train/test split, outside any pipeline?

*Answer:* Then mean and std would use test data too so we're leaking test into how we scale train. And when we actually deploy we never have "full dataset" — we only have what we had at train time. Fit scaler on train only so the evaluation is fair.

---


<span style="color:red">

# AI Usage Policy

- You are allowed to use AI tools to understand concepts or to debug syntax errors.

- You are not allowed to use AI to generate complete code or written responses.


- You must clearly declare any use of AI at the end of your notebook.

- Anyone found violating this policy will be awarded zero credit.

- A viva may also be conducted if misuse of AI is suspected.

</span>
